In [4]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))
sys.path.append(str(Path().resolve().parents[1]))



In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "key"

In [6]:
from __future__ import annotations

from typing import List, Literal
from pydantic import BaseModel, Field

from datetime import datetime

class TestCase(BaseModel):
    type: Literal["simple", "cross_log", "dynamic_preference"]
    task: str
    successful_response: str
    evidence: list[str]  # references to logs

class TestCaseList(BaseModel):
    test_cases: list[TestCase]

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", temperature=0, seed=42)
structured_llm = llm.with_structured_output(TestCaseList)

In [18]:
from langchain_core.messages import SystemMessage, HumanMessage

def generate_test_cases(persona, close_circle, messenger_logs, calendar_logs) -> TestCaseList:
    prompt_messages = [
        SystemMessage(
            content=(
                "You are generating benchmark test cases for evaluating an AI assistant's ability "
                "to retrieve and verify information from personal app logs.\n\n"

                "## Context\n"
                "You are given:\n"
                "1. A PERSONA (used only to make the requests realistic)\n"
                "2. A list of CLOSE SOCIAL CONNECTIONS\n"
                "3. APP LOGS (messages, calendar events, etc.)\n\n"

                "IMPORTANT:\n"
                "- The APP LOGS are the ONLY source of truth.\n"
                "- The PERSONA is ONLY for realism, tone, and plausibility.\n"
                "- The assistant being evaluated will ONLY have access to the APP LOGS at test time.\n"
                "- Do NOT rely on persona facts unless they are explicitly reflected in the logs.\n\n"

                "## Critical Framing Rule\n"
                "- Every test case task must be written as a NATURAL FIRST-PERSON request from the focal persona to an AI assistant.\n"
                "- The task must sound like something the focal persona would genuinely ask.\n"
                "- Do NOT write evaluator-style questions.\n"
                "- Do NOT mention logs, retrieval, evidence, benchmark, memory systems, or reasoning type in the task.\n"
                "- The retrieval challenge must be implicit.\n"
                "- The task should feel like a realistic help request about the persona's own life, plans, relationships, or commitments.\n\n"

                "## Your Task\n"
                "Generate EXACTLY 5 test cases for this persona.\n\n"

                "Each test case must include:\n"
                "- type\n"
                "- task\n"
                "- successful_response\n"
                "- evidence\n\n"

                "## Case Type Requirements\n"
                "### 1. Simple Fact-Check (1 case)\n"
                "- Internally solvable using ONLY ONE app log.\n"
                "- Task must still be a natural first-person request.\n\n"

                "### 2. Cross-Log Fact-Check (3 cases)\n"
                "- Internally must require combining information across multiple conversations or multiple apps.\n"
                "- Task must still be a natural first-person request.\n"
                "- The need for multi-log retrieval should be implicit, not stated.\n\n"

                "### 3. Dynamic Preference Tracking (1 case)\n"
                "- Internally must involve a change over time.\n"
                "- The correct answer must rely on the MOST RECENT logged information.\n"
                "- Task must still be a natural first-person request about a current preference, plan, or intention.\n\n"

                "## Successful Response Requirements\n"
                "- Must answer the user's request directly and naturally.\n"
                "- Must be grounded only in app logs.\n"
                "- Must explicitly cite the relevant logs.\n"
                "- Must explain why the cited evidence supports the answer.\n\n"

                "## Evidence Requirements\n"
                "- Include the exact logs or log snippets needed to justify the answer.\n"
                "- Include enough evidence to prove the answer and rule out weaker alternatives.\n\n"

                "## Style Requirements for task\n"
                "- Phrase tasks like: 'Can you remind me...', 'Did I ever say...', 'What did I end up deciding about...', "
                "'Am I free...', 'Who did I tell...', 'When was I planning to...'\n"
                "- Avoid sounding formal, synthetic, or evaluator-written.\n"
                "- Make each task feel personally motivated and socially realistic.\n\n"

                "Return structured output matching the schema."
            )
        ),
        HumanMessage(
            content="Persona:\n" + json.dumps(persona, indent=2, default=str)
        ),
        HumanMessage(
            content="Close Social Circle:\n" + json.dumps(close_circle, indent=2, default=str)
        ),
        HumanMessage(
            content="Messenger Logs:\n" + json.dumps(messenger_logs, indent=2, default=str)
        ),
        HumanMessage(
            content="Calendar Logs:\n" + json.dumps(calendar_logs, indent=2, default=str)
        ),
    ]

    result: TestCaseList = structured_llm.invoke(prompt_messages)
    return result

In [16]:
import os
import json

with open("../expand_and_socialize/expanded_personas/personas.json") as f:
    personas = json.load(f)

persona = personas[0]
close_circle = personas[1:]

with open("../app_log_gen/app_logs/transcripts.json") as f:
    messenger_logs = json.load(f)


with open("../app_log_gen/app_logs//calendar_logs.json") as f:
    calendar_logs = json.load(f)



In [19]:
result = generate_test_cases(persona, close_circle, messenger_logs, calendar_logs)

# Access structured output
for tc in result.test_cases:
    print(tc.type, tc.task)

simple Can you remind me what time Sara and I said we'd do our Sunday apartment reset?
cross_log Did I end up signing up for that spring 5K with Kelsey, and when was it?
cross_log What feedback did Danielle give me about moving toward management? I mostly remember bits and pieces.
cross_log What did I tell people about why I was trying to keep my weekends cheap?
dynamic_preference What did I end up deciding about classes or certifications for moving up at work? I know I went back and forth on that.


In [20]:
print (result.test_cases)

[TestCase(type='simple', task="Can you remind me what time Sara and I said we'd do our Sunday apartment reset?", successful_response='You and Sara said the Sunday apartment reset would be at 4 pm. In your messages with Sara on 2026-01-19, she suggested, “Maybe like 4 pm?” and you replied, “4 works.” That directly confirms the agreed time.', evidence=['Messenger log with Sara Jensen, 2026-01-19 02:27:00+00:00: Sara Jensen: “Exactly. Maybe like 4 pm? Late enough to be awake, early enough that it doesn’t ruin the evening.”', 'Messenger log with Sara Jensen, 2026-01-19 02:31:00+00:00: Mary Alberti: “4 works. I’ll make a tiny checklist and put it on the fridge so we don’t have to negotiate it every week.”']), TestCase(type='cross_log', task='Did I end up signing up for that spring 5K with Kelsey, and when was it?', successful_response='Yes — you did sign up for the spring 5K with Kelsey, and the race was on March 22, 2026. The messages show you first discussed the local April 5K, then you l

In [22]:
with open("./tasks/tasks.json", "w") as f:
    json.dump(
        [test_case.model_dump() for test_case in result.test_cases],
        f,
        indent=2,
        default=str
    )